## Why Fine-Tuning is Important

Fine-tuning is the process of taking a model that has already been trained on a massive, general dataset (like COCO) and performing additional training on a specific, smaller dataset.

### Key Benefits:
* **Specialization**: Learn specific nuances like 'License Plates'.
* **Efficiency**: Only needs a few hundred images and a fraction of the training time.
* **Transfer Learning**: Leverages pre-existing knowledge of edges and shapes.

# Fine-Tuning YOLOv8 with Roboflow

This notebook demonstrates how to:
1. **Setup**: Install the Ultralytics and Roboflow libraries.
2. **Data**: Download and visualize a dataset from Roboflow.
3. **Train**: Fine-tune a YOLOv8 model using Python and CLI.
4. **Evaluate**: Validate model performance.
5. **Predict**: Run inference on new data.
6. **Export**: Save weights for deployment.

In [ ]:
# Step 1: Install dependencies
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 21.0/107.7 GB disk)


In [ ]:
import ultralytics

ultralytics.checks()

### 1. Download Dataset from Roboflow
You will need your Roboflow API Key and the workspace/project details. You can find the 'Download' snippet in your Roboflow project dashboard under the 'Export' tab (select YOLOv8 format).

In [ ]:
# Step 2: Download Dataset
from roboflow import Roboflow

# Your project credentials
rf = Roboflow(api_key="Xopmh4f2LgI49DVpbEGh")
project = rf.workspace("bahria-university-ylifm").project("lisence-plate-k0bmy")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to lisence-plate-1 in yolov8:: 100%|██████████| 612/612 [00:00<00:00, 6575.39it/s]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.1 Visualize the Dataset
Before training, it's helpful to see our images and labels. Roboflow provides a `data.yaml` that tells YOLO where the images are and what the classes are.

In [ ]:
# Step 3: Visualize Ground Truth
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

train_images = glob.glob(f"{dataset.location}/train/images/*")[:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, img_path in enumerate(train_images):
    img = mpimg.imread(img_path)
    axes[i].imshow(img)
    axes[i].axis('off')
plt.suptitle("Sample Training Images from Roboflow")
plt.show()

### 2. Fine-Tune the YOLOv8 Model
We will use the pre-trained `yolov8n.pt` (nano) model as a starting point for better performance.

### Understanding Training Parameters

When training a YOLO model, several key parameters define how the learning process occurs:

*   **Task**: Defines the type of computer vision problem. `detect` is for Object Detection (finding objects with boxes), while others include `segment` (Instance Segmentation) and `classify` (Image Classification).
*   **Epochs**: One epoch is one full pass of the entire training dataset through the model. Increasing epochs can improve accuracy but may lead to 'overfitting' if too high.
*   **Image Size (imgsz)**: The resolution to which input images are resized before entering the network (e.g., 640x640). Higher resolutions can detect smaller objects but require more memory.
*   **Batch Size**: The number of images processed at once before the model's internal weights are updated. Smaller batches are easier on GPU memory, while larger batches can make training more stable.
*   **Confidence (conf)**: The threshold used during inference. Only predictions with a probability higher than this value will be shown.
*   **plots=True**: tells the training process to
generate visual graphs and images of training performance

In [ ]:
# Step 4: Fine-Tuning via Python API
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=25,
    imgsz=640,
    plots=True
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/lisence-plate-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

### Alternative: Training via YOLO CLI
You can also train models using the `yolo` command line interface. This is often faster to type for simple tasks.

In [ ]:
# Optional: Fine-Tuning via YOLO CLI
# This is an alternative to the code cell above
!yolo task=detect mode=train model=yolov8n.pt data={dataset.location}/data.yaml epochs=25 imgsz=640 batch=16

### 3. Validate and Predict

After training, you can evaluate the model performance and run inference on test images.

In [ ]:
# Step 5: Validation
# Evaluate the best model weights on the validation set
metrics = model.val()
print(f"mAP50-95: {metrics.box.map}")

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 703.1±249.9 MB/s, size: 42.1 KB)
val: Scanning /content/lisence-plate-1/valid/labels.cache... 58 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 58/58 16.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.9s/it 15.7s
                   all         58         66      0.886      0.818      0.911      0.532
Speed: 3.9ms preprocess, 251.1ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/runs/detect/val
mAP50-95: 0.5322322572310629


 ### 3.1 model.val() :

This runs validation (evaluation) on your model

What happens internally:

Loads your best trained weights (usually best.pt)
Runs inference on validation images
Compares predictions vs ground truth labels
Calculates performance metrics

💡 Important:

👉 No training happens here

👉 This is purely testing performance

### 3.2 Understanding the Results
When looking at the training output, keep an eye on these two:
1. **Loss (Box/Cls/DFL)**: Should decrease over time. If it goes up, the model is 'confused'.
2. **mAP50-95**: This is the 'Mean Average Precision'. Higher is better! It measures how accurately the model finds boxes and identifies classes.

### 4. Inference on New Images
You can use the `predict` method to run inference on images. The `save=True` argument will save the annotated images to the `runs/detect/predict` directory.

In [ ]:
# Step 6: Inference on Test Set
source_path = f"{dataset.location}/test/images"
results_list = model.predict(source=source_path, save=True, conf=0.40)
print(f"Predictions saved to: {results_list[0].save_dir}")


image 1/33 /content/lisence-plate-1/test/images/Cars10_png.rf.25bda693dd6bc1c6711ac868160068f2.jpg: 640x640 (no detections), 540.9ms
image 2/33 /content/lisence-plate-1/test/images/Cars12_png.rf.e44680984b9fd5fb5aa682bca5d4d09d.jpg: 640x640 1 licence, 310.0ms
image 3/33 /content/lisence-plate-1/test/images/Cars13_png.rf.a90811082872d3d17a7624ac25ebf726.jpg: 640x640 1 licence, 317.1ms
image 4/33 /content/lisence-plate-1/test/images/Cars140_png.rf.6dc19576ac5feb01cd0e5627e98e57b9.jpg: 640x640 1 licence, 326.8ms
image 5/33 /content/lisence-plate-1/test/images/Cars174_png.rf.0329563edd314ffc9b0b24c0fc40bc80.jpg: 640x640 1 licence, 343.9ms
image 6/33 /content/lisence-plate-1/test/images/Cars193_png.rf.595889e9269891261cef8c8b45e3ae51.jpg: 640x640 2 licences, 330.8ms
image 7/33 /content/lisence-plate-1/test/images/Cars195_png.rf.3a10f4f0bac05db9820a79bbc9a1b75a.jpg: 640x640 1 licence, 219.8ms
image 8/33 /content/lisence-plate-1/test/images/Cars204_png.rf.eed4304bdf9aba57f0f936289f203c72.jpg

### 5. Export and Save Model Weights
The model weights are already saved in the local runtime. You can export them to different formats or move them to a persistent location like Google Drive.

In [ ]:
# Step 7: Export Weights
model.export(format='onnx')
print(f"Final weights are available at: {model.ckpt_path}")

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 378ms
Prepared 4 packages in 1.95s
Installed 4 packages in 387ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.25.1
 + onnxslim==0.1.92

requirements: AutoUpdate success ✅ 3.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 5.7s, saved as '/content/runs/detect/tra

If you want to download the file directly to your computer from Colab, use the following snippet:

In [ ]:
from google.colab import files

# Download the best weights file
files.download(f'{results.save_dir}/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Understanding Google Colab for Model Training

When training models in Google Colab, it is important to understand how the environment manages resources:

### 1. Sessions and Runtimes
*   **Ephemeral Instances**: Colab provides a temporary virtual machine. If you are inactive for an extended period or close the tab, the session may time out and the runtime will be recycled.
*   **Session Limits**: Standard sessions typically last up to 12 hours. Any files not saved to Google Drive or exported (like our model weights) will be deleted when the session ends.

### 2. Hardware Acceleration
*   **GPU/TPU**: For deep learning tasks like YOLOv8 training, Colab offers free access to GPUs (like the T4). You can check your current hardware via `Edit > Notebook settings`.
*   **CPU vs. GPU**: Training on a CPU is significantly slower. Always ensure a GPU is active for 'Fine-Tuning' tasks to reduce training time from hours to minutes.

### 3. File Persistence
*   **Local Storage**: The `/content` folder is local to the current session.
*   **Google Drive Integration**: To keep your datasets and weights permanently, you can 'Mount Google Drive' using the folder icon on the left or by running `from google.colab import drive; drive.mount('/content/drive')`.